# GraphPart train/validation/test split, phosphatase

This notebook runs the split on a Colab VM, where the time limit is hours rather than minutes.

It installs EMBOSS and GraphPart pinned to the audited commit, reheaders the FASTA, runs the split,
checks the result, and downloads everything. Nothing is written outside `/content`.

## Before you start

1. Open this notebook in Google Colab.
2. **Runtime > Change runtime type > Python 3 > CPU.** GraphPart does not use a GPU. Asking for one
   only makes the runtime harder to get.
3. Leave the config cell alone for the first run. Every value in it is provisional and marked.
4. Run every cell in order. Stop when a cell raises an error. Do not edit a check to make it pass.
5. When prompted, upload `data/phosphatase/phosphatase/tmp/sequences.fasta`. A SHA256 check rejects
   any other file.
6. **Keep the browser tab open.** Colab disconnects idle sessions and you lose the alignment work.
7. Expect this to take a while. See the runtime estimate in section 4 before you start worrying.

## What this uploads

The 1,587 phosphatase sequences go to a Google Colab VM.

In [ ]:
ALIGNMENT_MODE = "needle"
THRESHOLD      = 0.3
TEST_RATIO     = 0.1
VAL_RATIO      = 0.1
DENOMINATOR    = "full"
TRANSFORMATION = "one-minus"
TRIANGULAR     = True
LABELS_NAME    = None
DEDUPLICATE    = True

GRAPHPART_COMMIT = "f470726"
EXPECTED_RECORDS = 1587
EXPECTED_UNIQUE  = 1480
EXPECTED_GROUPS  = 56
EXPECTED_SHA256  = "c806be15f5eb32c2fe5fdf89c1174df2e8b0fa2b43873d19a86516c03e71149f"

if round(TEST_RATIO * 100) % 5 or round(VAL_RATIO * 100) % 5:
    raise ValueError("GraphPart only accepts ratios that are multiples of 0.05")
if ALIGNMENT_MODE not in ("needle", "mmseqs2"):
    raise ValueError("This notebook covers the needle and mmseqs2 modes only")

print(f"mode={ALIGNMENT_MODE} threshold={THRESHOLD} test={TEST_RATIO} val={VAL_RATIO}")
print("deduplicate:", DEDUPLICATE)
print("partitions GraphPart will use:",
      10 if (round(TEST_RATIO*100) % 10 == 0 and round(VAL_RATIO*100) % 10 == 0) else 20)

## 1. Check the runtime and install

EMBOSS comes from the Ubuntu archive and supplies `needleall`. GraphPart is installed from the
pinned commit rather than from PyPI, so the version is fixed.

In [ ]:
import os, sys, subprocess, shutil, time, uuid, hashlib, json
from pathlib import Path

if "google.colab" not in sys.modules:
    raise RuntimeError("Run this notebook in Google Colab, not against a local kernel")

t0 = time.time()
if ALIGNMENT_MODE == "needle":
    subprocess.run("apt-get -qq update && apt-get -qq install -y emboss",
                   shell=True, check=True)
else:
    subprocess.run("apt-get -qq update && apt-get -qq install -y mmseqs2",
                   shell=True, check=True)

subprocess.run(
    f"pip -q install git+https://github.com/graph-part/graph-part@{GRAPHPART_COMMIT}",
    shell=True, check=True)

binary = "needleall" if ALIGNMENT_MODE == "needle" else "mmseqs"
for tool in (binary, "graphpart"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"{tool} is not on PATH after install. Stop and read the output above")

print("install took", round(time.time() - t0), "s")
print(binary, "->", shutil.which(binary))
print("cpus:", os.cpu_count())

## 2. Upload the FASTA and verify it

The SHA256 check exists because "sequences.fasta" is a name several files in this project could
have. A byte-for-byte match is the only way to know which one arrived.

In [ ]:
from google.colab import files

ROOT = Path("/content") / f"graphpart_{uuid.uuid4().hex[:12]}"
ROOT.mkdir(parents=True)

print("Select sequences.fasta")
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError(f"Upload exactly one file, got {len(uploaded)}")

name, payload = next(iter(uploaded.items()))
digest = hashlib.sha256(payload).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(
        f"Wrong file. Expected sha256 {EXPECTED_SHA256}, got {digest}. "
        "This is not the registered phosphatase FASTA")

RAW_FASTA = ROOT / "sequences.fasta"
RAW_FASTA.write_bytes(payload)

n_records = payload.decode("utf-8").count("\n>") + payload.decode("utf-8").startswith(">")
if n_records != EXPECTED_RECORDS:
    raise RuntimeError(f"Expected {EXPECTED_RECORDS} records, found {n_records}")

print(f"{name}: {len(payload)} bytes, {n_records} records, sha256 verified")

## 3. Reheader and deduplicate

**This is not cosmetic. GraphPart cannot read the file without it.**

GraphPart takes the identifier to be everything before the first `|` in the header
(`graph_part/needle_utils.py:50`), because the pipes are where it expects `key=value` metadata.
Our headers are UniProt style:

```
>sp|A0A0G2JXT6|MTMR6_RAT Phosphatidylinositol-3,5-bisphosphate 3-phosphatase MTMR6 OS=...
```

Split on `|` and take the first field, and all 1,587 records become the single token `sp`. So we
rewrite each header to its accession, `>A0A0G2JXT6`, which is unique across all 1,587 records.

The original file is kept untouched beside the rewritten one, so the mapping stays recoverable.

### Why deduplicate first

Phosphatase has 1,480 unique sequences across 1,587 records, in 56 repeated-sequence groups
covering 163 records. Identical sequences sit at 100% identity, so distance 0,
below any threshold we would ever set. GraphPart therefore either puts a duplicate pair in the same
partition or deletes one of them, and which of those happens is not controlled.

That matters for reading the result. If duplicates get deleted, the removal count mixes duplicate
cleanup with homology separation, and the retention figure stops being comparable with the paper's
94.21% to 99.99%.

So we split the 1,480 representatives, then map every duplicate record back to whichever set its
representative landed in. All 1,587 records still get assigned, no duplicate can straddle a
boundary, the removal count means only what it says, and there are about 13% fewer pairs to align.

The representative is the lowest accession in each group, which is the same rule `src/data.py`
already uses, so this representative set matches the one the rest of the project works with.

Set `DEDUPLICATE = False` in the config to split all 1,587 instead and compare.

In [ ]:
import re

SPLIT_FASTA = ROOT / "sequences_graphpart.fasta"
mapping = {}
records = []

accession, chunks = None, []
with RAW_FASTA.open(encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(">"):
            if accession is not None:
                records.append((accession, "".join(chunks)))
            parts = line[1:].strip().split("|")
            if len(parts) < 3:
                raise RuntimeError(f"Header is not UniProt style, cannot reheader: {line[:80]}")
            accession = parts[1]
            if not re.fullmatch(r"[A-Z0-9]+", accession):
                raise RuntimeError(f"Accession looks wrong: {accession!r}")
            mapping[accession] = line[1:].strip()
            chunks = []
        else:
            chunks.append(line.strip())
if accession is not None:
    records.append((accession, "".join(chunks)))

if len(mapping) != EXPECTED_RECORDS:
    raise RuntimeError(
        f"{EXPECTED_RECORDS} records in, {len(mapping)} unique accessions out. "
        "Duplicate accessions would silently merge records")

from collections import defaultdict

by_sequence = defaultdict(list)
for accession, sequence in records:
    by_sequence[sequence].append(accession)

representative_of = {}
for sequence, members in by_sequence.items():
    rep = min(members)
    for member in members:
        representative_of[member] = rep

groups_with_duplicates = sum(1 for m in by_sequence.values() if len(m) > 1)
records_in_those = sum(len(m) for m in by_sequence.values() if len(m) > 1)

print(f"{len(records)} records, {len(by_sequence)} unique sequences, "
      f"{groups_with_duplicates} groups with duplicates covering {records_in_those} records")

if len(by_sequence) != EXPECTED_UNIQUE or groups_with_duplicates != EXPECTED_GROUPS:
    raise RuntimeError(
        f"Expected {EXPECTED_UNIQUE} unique sequences in {EXPECTED_GROUPS} duplicate groups, "
        f"got {len(by_sequence)} and {groups_with_duplicates}. This FASTA is not the one the "
        "data card describes")

if DEDUPLICATE:
    keep = {min(members) for members in by_sequence.values()}
    written = [(a, s) for a, s in records if a in keep]
else:
    written = records

with SPLIT_FASTA.open("w", encoding="utf-8") as fh:
    for accession, sequence in written:
        fh.write(f">{accession}\n")
        for i in range(0, len(sequence), 60):
            fh.write(sequence[i:i + 60] + "\n")

SUBMITTED = len(written)
PAIRS = SUBMITTED * (SUBMITTED - 1) // 2

(ROOT / "header_map.json").write_text(json.dumps(mapping, indent=1), encoding="utf-8")
(ROOT / "representative_of.json").write_text(
    json.dumps(representative_of, indent=1), encoding="utf-8")

print(f"submitting {SUBMITTED} sequences to GraphPart, {PAIRS:,} pairs to align")

## 4. Run GraphPart

**Rough runtime estimate.** Deduplicated, 1,480 sequences give 1,094,460 unordered pairs against
1,258,491 for the full 1,587, so about 13% less work. Mean length is 494 residues, so a typical
Needleman-Wunsch alignment fills about 244,000 cells, and the whole job is order 3e11 cells. On a couple of Colab cores that is tens of minutes, not hours, but it is an
estimate from lengths and not a measurement. If it is still running after two hours, stop it and
switch `ALIGNMENT_MODE` to `"mmseqs2"`.

`--save-checkpoint-path` writes the computed similarities. If you later re-run at a **higher**
threshold you can reuse them through `precomputed` mode instead of aligning again. A lower
threshold needs a fresh run, because everything above the original threshold was discarded.

In [ ]:
OUT_CSV    = ROOT / "graphpart_result.csv"
CHECKPOINT = ROOT / "similarities.checkpoint"
LOG_PATH   = ROOT / "graphpart_run.log"

cmd = [
    "graphpart", ALIGNMENT_MODE,
    "--fasta-file", str(SPLIT_FASTA),
    "--threshold", str(THRESHOLD),
    "--out-file", str(OUT_CSV),
    "--transformation", TRANSFORMATION,
    "--test-ratio", str(TEST_RATIO),
    "--val-ratio", str(VAL_RATIO),
    "--denominator", DENOMINATOR,
    "--save-checkpoint-path", str(CHECKPOINT),
]
if ALIGNMENT_MODE == "needle":
    cmd += ["--threads", str(os.cpu_count() or 2), "--chunks", str((os.cpu_count() or 2) * 5)]
    if TRIANGULAR:
        cmd += ["--triangular"]
if LABELS_NAME:
    cmd += ["--labels-name", LABELS_NAME]

print(" ".join(cmd), "\n")

start = time.time()
with LOG_PATH.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    proc.wait()
elapsed = time.time() - start

print(f"\nexit code {proc.returncode} after {elapsed/60:.1f} minutes")
if proc.returncode != 0:
    raise RuntimeError(
        "GraphPart failed. If the message names lost partitions, the threshold is too strict for "
        "this dataset and that is itself a finding worth recording")

## 5. Check the result

Four checks. Each one can fail, and a failure here is information rather than an inconvenience.

In [ ]:
import pandas as pd

result = pd.read_csv(OUT_CSV)
print(result.head(), "\n")
print("columns:", list(result.columns), "\n")

label_col = "cluster" if "cluster" in result.columns else result.columns[-1]
sizes = result[label_col].value_counts().sort_index()

retained = len(result)
removed  = SUBMITTED - retained
print(f"submitted {SUBMITTED}, retained {retained}, removed {removed} "
      f"({removed / SUBMITTED:.1%})\n")
print("partition sizes:"); print(sizes, "\n")

assert retained + removed == SUBMITTED, "records do not reconcile"

if len(sizes) < 3:
    raise RuntimeError(f"Expected train, validation and test, got {len(sizes)} sets")

id_col = result.columns[0]
assert result[id_col].is_unique, "duplicate identifiers in the output"
unknown = set(result[id_col]) - set(mapping)
assert not unknown, f"output holds identifiers we never supplied: {list(unknown)[:5]}"

print(f"retention {retained / SUBMITTED:.2%}. "
      "The paper kept 94.21% to 99.99% on its protein sets, Table 1 p. 4. "
      "Much lower here means phosphatase is more redundant than those, which is a result, "
      "not a bug.")

## 6. Map duplicates back to their representative's set

Only the representatives were split. This puts every duplicate record into whichever set its
representative landed in, so the assignment covers all 1,587 records again.

A duplicate whose representative GraphPart removed is removed too. That is the honest treatment:
it is the same sequence, so it carries the same leakage risk.

With `DEDUPLICATE = False` this is a no-op and the counts pass through unchanged.

In [ ]:
assignment = dict(zip(result[id_col], result[label_col]))

expanded_rows, dropped = [], []
for accession in mapping:
    rep_accession = representative_of[accession]
    if rep_accession in assignment:
        expanded_rows.append({
            "accession": accession,
            "representative": rep_accession,
            "is_representative": accession == rep_accession,
            "set": assignment[rep_accession],
        })
    else:
        dropped.append(accession)

expanded = pd.DataFrame(expanded_rows)
EXPANDED_CSV = ROOT / "graphpart_assignment_all_records.csv"
expanded.to_csv(EXPANDED_CSV, index=False)

if len(expanded) + len(dropped) != EXPECTED_RECORDS:
    raise RuntimeError("expanded assignment does not cover all records")

print(f"assigned {len(expanded)} of {EXPECTED_RECORDS} records, {len(dropped)} dropped "
      f"because their representative was removed\n")
print(expanded["set"].value_counts().sort_index(), "\n")
print("duplicates folded in:", int((~expanded["is_representative"]).sum()))

straddling = expanded.groupby("representative")["set"].nunique()
assert (straddling == 1).all(), "a sequence ended up in more than one set"
print("no sequence spans two sets: confirmed")

## 7. Download everything

Take all seven files. The log and the config record are what make the run quotable later; the
checkpoint is the expensive part you do not want to recompute.

In [ ]:
config_record = {
    "run_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "graphpart_commit": GRAPHPART_COMMIT,
    "alignment_mode": ALIGNMENT_MODE,
    "threshold": THRESHOLD,
    "test_ratio": TEST_RATIO,
    "val_ratio": VAL_RATIO,
    "denominator": DENOMINATOR,
    "transformation": TRANSFORMATION,
    "triangular": TRIANGULAR,
    "labels_name": LABELS_NAME,
    "input_sha256": EXPECTED_SHA256,
    "deduplicated": DEDUPLICATE,
    "records_in": EXPECTED_RECORDS,
    "sequences_submitted": SUBMITTED,
    "sequences_retained": int(retained),
    "sequences_removed": int(removed),
    "records_assigned_after_expansion": int(len(expanded)),
    "elapsed_minutes": round(elapsed / 60, 2),
    "command": " ".join(cmd),
    "cpu_count": os.cpu_count(),
}
CONFIG_PATH = ROOT / "run_config.json"
CONFIG_PATH.write_text(json.dumps(config_record, indent=2), encoding="utf-8")
print(json.dumps(config_record, indent=2))

for path in (OUT_CSV, EXPANDED_CSV, LOG_PATH, CONFIG_PATH,
             ROOT / "header_map.json", ROOT / "representative_of.json", CHECKPOINT):
    if path.exists():
        files.download(str(path))
    else:
        print("missing, not downloaded:", path.name)

## After the run

Put the downloaded files in `src/data-split/graphpart/` beside this notebook.

In [ ]:
import shutil
from google.colab import files

archive_name = ROOT.name
shutil.make_archive(archive_name, 'zip', ROOT)

print(f'Zipping {ROOT} to {archive_name}.zip')
files.download(f'{archive_name}.zip')

In [ ]:
THRESHOLD_FLOOR = 0.20
SWEEP = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
DENOMINATORS = ["full", "shortest"]

import subprocess, time, json, os
from pathlib import Path
import pandas as pd

SWEEP_DIR = ROOT / "sweep"
SWEEP_DIR.mkdir(exist_ok=True)

for required in ("SPLIT_FASTA", "SUBMITTED", "THRESHOLD", "TEST_RATIO", "VAL_RATIO"):
    if required not in globals():
        raise RuntimeError(
            f"{required} is missing. These cells continue the earlier session; do not restart "
            "the runtime before running them")

def run_graphpart(threshold, edge_file=None, denominator=None, tag=""):
    out = SWEEP_DIR / f"result_{tag}.csv"
    cmd = ["graphpart", "precomputed" if edge_file else "needle",
           "--fasta-file", str(SPLIT_FASTA),
           "--threshold", str(threshold),
           "--out-file", str(out),
           "--transformation", "one-minus",
           "--test-ratio", str(TEST_RATIO), "--val-ratio", str(VAL_RATIO)]
    if edge_file:
        cmd += ["--edge-file", str(edge_file), "--metric-column", "2"]
    else:
        cmd += ["--denominator", denominator,
                "--save-checkpoint-path", str(SWEEP_DIR / f"edges_{denominator}.checkpoint"),
                "--threads", str(os.cpu_count() or 2),
                "--chunks", str((os.cpu_count() or 2) * 5), "--triangular"]

    started = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - started
    (SWEEP_DIR / f"log_{tag}.txt").write_text(proc.stdout + proc.stderr, encoding="utf-8")

    if proc.returncode != 0:
        return {"threshold": threshold, "ok": False, "elapsed_s": round(elapsed, 1),
                "note": proc.stdout.strip().splitlines()[-1] if proc.stdout.strip() else "failed"}

    result = pd.read_csv(out)
    sizes = result["cluster"].value_counts().sort_index()
    return {"threshold": threshold, "ok": True, "elapsed_s": round(elapsed, 1),
            "retained": len(result), "removed": SUBMITTED - len(result),
            "retention": len(result) / SUBMITTED,
            "sets": len(sizes), "largest": int(sizes.max()), "smallest": int(sizes.min()),
            "result_path": str(out)}

print("helpers ready. sweep dir:", SWEEP_DIR)

In [ ]:
checkpoints = {}
for denominator in DENOMINATORS:
    path = SWEEP_DIR / f"edges_{denominator}.checkpoint"
    if path.exists():
        print(f"{denominator}: checkpoint already present, skipping the alignment")
    else:
        print(f"{denominator}: aligning at threshold {THRESHOLD_FLOOR}, this is the slow step")
        info = run_graphpart(THRESHOLD_FLOOR, denominator=denominator, tag=f"floor_{denominator}")
        print("   ", info)
    checkpoints[denominator] = path
    edges = pd.read_csv(path, header=None, names=["a", "b", "identity"])
    total_pairs = SUBMITTED * (SUBMITTED - 1) // 2
    print(f"    {len(edges):,} edges at or above {THRESHOLD_FLOOR} "
          f"({100 * len(edges) / total_pairs:.2f}% of {total_pairs:,} pairs), "
          f"identity {edges.identity.min():.3f} to {edges.identity.max():.3f}\n")

In [ ]:
rows = []
for denominator, path in checkpoints.items():
    edges = pd.read_csv(path, header=None, names=["a", "b", "identity"])
    for threshold in SWEEP:
        if threshold < THRESHOLD_FLOOR:
            continue
        info = run_graphpart(threshold, edge_file=path,
                             tag=f"{denominator}_{str(threshold).replace('.', 'p')}")
        info["denominator"] = denominator
        info["edges_at_threshold"] = int((edges.identity >= threshold).sum())
        rows.append(info)
        print(f"{denominator:9} {threshold:.2f}  "
              f"edges {info['edges_at_threshold']:>7,}  "
              + (f"removed {info['removed']:>4}  retention {info['retention']:.2%}"
                 if info["ok"] else f"FAILED: {info['note'][:60]}"))

sweep = pd.DataFrame(rows)
sweep.to_csv(SWEEP_DIR / "threshold_sweep.csv", index=False)
print("\nwritten:", SWEEP_DIR / "threshold_sweep.csv")

In [ ]:
pivot = sweep.pivot_table(index="threshold", columns="denominator",
                          values=["edges_at_threshold", "removed", "retention"])
print(pivot.to_string(), "\n")

try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for denominator in DENOMINATORS:
        part = sweep[(sweep.denominator == denominator) & sweep.ok]
        axes[0].plot(part.threshold, part.edges_at_threshold, marker="o", label=denominator)
        axes[1].plot(part.threshold, part.removed, marker="o", label=denominator)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("identity threshold"); axes[0].set_ylabel("edges above threshold")
    axes[0].set_title("How many pairs count as too similar")
    axes[1].set_xlabel("identity threshold"); axes[1].set_ylabel("sequences removed")
    axes[1].set_title("What separation costs")
    for ax in axes:
        ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(SWEEP_DIR / "threshold_sweep.png", dpi=150)
    plt.show()
except ImportError:
    print("matplotlib not available, table only")

first_bite = sweep[sweep.ok & (sweep.removed > 0)].sort_values("threshold")
if len(first_bite):
    row = first_bite.iloc[0]
    print(f"\nFirst threshold that removes anything: {row.threshold} under {row.denominator}, "
          f"{int(row.removed)} sequences, retention {row.retention:.2%}")
else:
    print(f"\nNothing was removed at any threshold down to {THRESHOLD_FLOOR}. "
          "Lower THRESHOLD_FLOOR and re-run step A, or accept that phosphatase is this diverse "
          "under both denominators and say so.")

In [ ]:
AUDIT_SPLIT = "full"
AUDIT_RULE  = "shortest"

candidates = sweep[(sweep.denominator == AUDIT_SPLIT) & sweep.ok]
candidates = candidates[(candidates.threshold - THRESHOLD).abs() < 1e-9]
if candidates.empty:
    raise RuntimeError(
        f"No successful {AUDIT_SPLIT} run at threshold {THRESHOLD}. "
        f"Add {THRESHOLD} to SWEEP and re-run step B")
split_result = pd.read_csv(candidates.iloc[0]["result_path"])
partition_of = dict(zip(split_result["AC"], split_result["cluster"]))

audit = pd.read_csv(checkpoints[AUDIT_RULE], header=None, names=["a", "b", "identity"])
audit["pa"] = audit.a.map(partition_of)
audit["pb"] = audit.b.map(partition_of)
cross = audit[audit.pa.notna() & audit.pb.notna() & (audit.pa != audit.pb)]

violations = cross[cross.identity >= THRESHOLD]
print(f"split built at {THRESHOLD} under '{AUDIT_SPLIT}', audited under '{AUDIT_RULE}'")
print(f"cross-partition pairs at or above {THRESHOLD}: {len(violations)}")
if len(violations):
    print(f"worst: {violations.identity.max():.3f}\n")
    print(violations.sort_values("identity", ascending=False)
          .head(10)[["a", "b", "identity", "pa", "pb"]].to_string(index=False))
    print("\nThese are pairs the split treats as unrelated and the other denominator does not. "
          "That is the honest limit of what this split guarantees, and it belongs in the write-up.")
else:
    print("\nNone. The split holds under both denominators, which is a stronger claim than the "
          "20 September run could make on its own.")

## After the sweep

Download `sweep/threshold_sweep.csv`, `sweep/threshold_sweep.png` and both `.checkpoint` files. The
checkpoints are the expensive part and they make any future threshold at or above the floor free.

In [ ]:
import datetime
print("Current time:", datetime.datetime.now())

In [ ]:
import shutil, subprocess, os, time
import pandas as pd

MMSEQS_FLOOR = 0.20
MMSEQS_EDGES = SWEEP_DIR / "edges_mmseqs2.checkpoint"

if shutil.which("mmseqs") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y mmseqs2", shell=True, check=False)
if shutil.which("mmseqs") is None:
    raise RuntimeError(
        "mmseqs not on PATH. Try `!conda install -y -c conda-forge -c bioconda mmseqs2`, "
        "or `!pip install mmseqs2` if that fails, then re-run this cell")
if shutil.which("mmseqs_fake_prefilter.sh") is None:
    raise RuntimeError(
        "mmseqs_fake_prefilter.sh is missing. GraphPart needs it to force all-vs-all; without it "
        "MMseqs2 prefilters and drops pairs silently. Reinstall graph-part")

if MMSEQS_EDGES.exists():
    print("edge list already present, skipping")
else:
    cmd = ["graphpart", "mmseqs2",
           "--fasta-file", str(SPLIT_FASTA),
           "--threshold", str(MMSEQS_FLOOR),
           "--out-file", str(SWEEP_DIR / "result_mmseqs2_floor.csv"),
           "--transformation", "one-minus",
           "--test-ratio", str(TEST_RATIO), "--val-ratio", str(VAL_RATIO),
           "--denominator", "shortest",
           "--save-checkpoint-path", str(MMSEQS_EDGES)]
    print(" ".join(cmd), "\n")
    started = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True)
    (SWEEP_DIR / "log_mmseqs2_floor.txt").write_text(proc.stdout + proc.stderr, encoding="utf-8")
    print(f"exit {proc.returncode} after {(time.time() - started) / 60:.1f} min")
    if proc.returncode != 0:
        print("partitioning failed, which does not matter here. Checking the edge list survived.")

if not MMSEQS_EDGES.exists():
    raise RuntimeError("No MMseqs2 edge list was written. Read log_mmseqs2_floor.txt")

mm = pd.read_csv(MMSEQS_EDGES, header=None, names=["a", "b", "identity"])
TOTAL_PAIRS = SUBMITTED * (SUBMITTED - 1) // 2
print(f"\n{len(mm):,} MMseqs2 edges at or above {MMSEQS_FLOOR} "
      f"({100 * len(mm) / TOTAL_PAIRS:.2f}% of {TOTAL_PAIRS:,} pairs), "
      f"identity {mm.identity.min():.3f} to {mm.identity.max():.3f}")
print("\nfor comparison, needle at the same floor:")
for name, path in checkpoints.items():
    n = sum(1 for _ in open(path))
    print(f"  {name:9} {n:>9,}  ({100 * n / TOTAL_PAIRS:.2f}%)")

In [ ]:
def count_violations(result_csv, edge_file, threshold):
    res = pd.read_csv(result_csv)
    partition = dict(zip(res["AC"], res["cluster"]))
    edges = pd.read_csv(edge_file, header=None, names=["a", "b", "identity"])
    pa = edges.a.map(partition)
    pb = edges.b.map(partition)
    cross = edges[pa.notna() & pb.notna() & (pa != pb)]
    hit = cross[cross.identity >= threshold - 1e-9]
    return len(hit), (hit.identity.max() if len(hit) else 0.0)

rules = dict(checkpoints)
rules["mmseqs2"] = MMSEQS_EDGES

audit_rows = []
for _, run in sweep[sweep.ok].iterrows():
    row = {"split": f"{run.denominator} @ {run.threshold}",
           "removed": int(run.removed),
           "smallest_set": int(run.smallest)}
    for rule_name, rule_path in rules.items():
        n, worst = count_violations(run.result_path, rule_path, run.threshold)
        row[f"{rule_name}_pairs"] = n
        row[f"{rule_name}_worst"] = round(float(worst), 3)
    audit_rows.append(row)
    print(row)

audit_table = pd.DataFrame(audit_rows)
audit_table.to_csv(SWEEP_DIR / "audit_matrix.csv", index=False)
print("\n", audit_table.to_string(index=False))

clean = audit_table[(audit_table.mmseqs2_pairs == 0) & (audit_table.smallest_set >= 100)]
print("\nCandidates with no MMseqs2 violation and a usable smallest set (>=100):")
print(clean.to_string(index=False) if len(clean) else
      "  none. Every candidate either leaks under MMseqs2 or has a set too small to use. "
      "That is a result: report it rather than picking the least bad one quietly.")

## What to download

Four files, all in the `sweep` folder:

1. **`audit_matrix.csv`** - the main result. One row per candidate split, violation counts under
   every rule.
2. **`edges_mmseqs2.checkpoint`** - the MMseqs2 edge list. Expensive to regenerate and reusable for
   any future audit at or above 0.20.
3. **`log_mmseqs2_floor.txt`** - the MMseqs2 run log, including whether its own partitioning
   failed and how many edges it found.
4. **`threshold_sweep.csv`** - again if it changed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

mm_edges = pd.read_csv(
    MMSEQS_EDGES, header=None, names=["a", "b", "identity"]
)

mm_rows = []
for threshold in SWEEP:
    if threshold < MMSEQS_FLOOR:
        continue

    info = run_graphpart(
        threshold,
        edge_file=MMSEQS_EDGES,
        tag=f"mmseqs2_{str(threshold).replace('.', 'p')}",
    )
    info["denominator"] = "mmseqs2"
    info["edges_at_threshold"] = int(
        (mm_edges.identity >= threshold).sum()
    )
    mm_rows.append(info)

    if info["ok"]:
        print(
            f"MMseqs2 {threshold:.2f}: removed {info['removed']}, "
            f"retention {info['retention']:.2%}"
        )
    else:
        print(f"MMseqs2 {threshold:.2f}: FAILED — {info['note']}")

combined_sweep = pd.concat(
    [sweep, pd.DataFrame(mm_rows)], ignore_index=True
)
csv_path = SWEEP_DIR / "threshold_sweep_with_mmseqs2.csv"
combined_sweep.to_csv(csv_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = {
    "full": "NW / full",
    "shortest": "NW / shortest",
    "mmseqs2": "MMseqs2 / shortest",
}

for method, label in labels.items():
    part = combined_sweep[
        combined_sweep["denominator"] == method
    ].sort_values("threshold")
    successful = part[part["ok"]]

    axes[0].plot(
        part["threshold"], part["edges_at_threshold"],
        marker="o", label=label,
    )
    axes[1].plot(
        successful["threshold"], successful["retention"] * 100,
        marker="o", label=label,
    )

axes[0].set_yscale("log")
axes[0].set_ylabel("Number of pairs")
axes[0].set_title("Sequence pairs with identity ≥ cutoff")
axes[1].set_ylabel("Unique sequences retained (%)")
axes[1].set_title("Sequences retained after splitting")

for ax in axes:
    ax.set_xlabel("Minimum sequence identity (%)")
    ax.legend()
    ax.grid(alpha=0.3)

fig.tight_layout()
png_path = SWEEP_DIR / "threshold_sweep_with_mmseqs2.png"
fig.savefig(png_path, dpi=200, bbox_inches="tight")
plt.show()

files.download(str(csv_path))
files.download(str(png_path))